# Kaggle llama.cpp backend runner

Notebook pendek untuk menjalankan backend dari repo GitHub.

Urutan: dependency/prebuilt, download model, start server, start ngrok.

## Cell 1 — Dependency + prebuilt llama.cpp CUDA

Ubah `REPO_URL` ke repo GitHub kamu setelah repo ini di-push.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/kaggle-llamacpp-backend.git"
REPO_DIR = Path("/kaggle/working/kaggle-llamacpp-backend")

if not REPO_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Ubah REPO_URL dulu ke repo GitHub kamu.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

from kaggle_llamacpp import ensure_llamacpp_cuda, print_status

state = ensure_llamacpp_cuda()
state


## Cell 2 — Download model GGUF

Download memakai `requests` + `tqdm`, bukan aria2, supaya progress lebih realtime di Kaggle.

In [ ]:
from kaggle_llamacpp import download_model

MODEL_URL = "https://huggingface.co/mradermacher/gemma-4-26B-A4B-Heretic-Stable-i1-GGUF/resolve/main/gemma-4-26B-A4B-Heretic-Stable.i1-Q4_K_M.gguf"

model_path = download_model(MODEL_URL)
model_path


## Cell 3 — Config + start llama-server + test response

Semua setting runtime ditaruh di sini, bukan di downloader.

In [ ]:
from kaggle_llamacpp import ServerConfig, start_llama_server, wait_until_ready, test_chat_completion

cfg = ServerConfig(
    ctx_size=2048,
    batch_size=256,
    ubatch_size=64,
    port=8080,
    gpu_layers=999,
    split_mode="layer",
    tensor_split="1,1",
    threads=4,
    parallel=1,
    reasoning_format="none",
    enable_thinking=False,
)

pid = start_llama_server(cfg)

ready = wait_until_ready(port=cfg.port, timeout_s=900)
if not ready:
    raise RuntimeError("Server belum ready atau proses mati. Cek log di output.")

text = test_chat_completion(
    "Say hello in one short sentence. Do not explain your reasoning.",
    port=cfg.port,
    max_tokens=64,
)
print("FINAL:", text)


## Cell 4 — Start ngrok tunnel

Sebelum menjalankan cell ini, buat Kaggle Secret bernama `NGROK_AUTHTOKEN`.

In [ ]:
from kaggle_llamacpp import start_ngrok_tunnel

public_url = start_ngrok_tunnel(
    port=8080,
    secret_name="NGROK_AUTHTOKEN",
)

print("Use this as OpenAI-compatible Base URL:")
print(public_url + "/v1")


## Optional — status dan cleanup

In [ ]:
from kaggle_llamacpp import print_status
print_status(lines=120)


In [ ]:
from kaggle_llamacpp import stop_llama_server, stop_ngrok_tunnel

# Uncomment kalau ingin stop manual.
# stop_ngrok_tunnel()
# stop_llama_server()
